# Xenomorfo-Q: casos de prueba de la directora

Este notebook ejecuta los cinco casos C1–C5 con semilla fija y presenta la pregunta, el resultado esperado, la respuesta del modelo y el registro completo de herramientas. Cada caso usa una partida nueva. Los resultados se guardan en JSON y Markdown para revisarlos posteriormente; la valoración de la respuesta es manual.

**Ejecución:** abre el notebook desde la raíz del repositorio en Jupyter, o ábrelo en Google Colab y ejecuta las celdas en orden. Las llamadas al modelo requieren conexión y pueden consumir cuota.

## 1. Preparar el repositorio

En local abre Jupyter desde la raíz del proyecto. Si ya tienes una copia en otra ubicación, define la variable de entorno `XQ_RUTA_REPOSITORIO` con su ruta antes de iniciar Jupyter. En Colab, si no hay una copia disponible, se clona el repositorio público en `/content/Xenomorfo-Q`. El clon usa la rama predeterminada y **no incluye cambios locales sin publicar**: para reproducir este entregable, asegúrate de que la copia contenga el soporte para `XQ_PROVEEDOR` y los casos actuales; utiliza la rama o commit correspondiente una vez publicados.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

ruta_configurada = os.environ.get('XQ_RUTA_REPOSITORIO')
raiz = Path(ruta_configurada).expanduser().resolve() if ruta_configurada else Path.cwd()
if EN_COLAB and not ruta_configurada and not (raiz / 'experiments' / 'casos.py').is_file():
    raiz = Path('/content/Xenomorfo-Q')
    if not raiz.exists():
        subprocess.run(['git', 'clone', 'https://github.com/jalvarez01/Xenomorfo-Q.git', str(raiz)], check=True)

if not (raiz / 'experiments' / 'casos.py').is_file() or not (raiz / 'requirements.txt').is_file():
    raise FileNotFoundError('No se encontró el proyecto o experiments/casos.py. Indica una copia completa mediante XQ_RUTA_REPOSITORIO y reinicia el notebook.')
os.chdir(raiz)
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
print('Repositorio:', raiz)

## 2. Instalar dependencias

La siguiente celda instala los requisitos del proyecto en el intérprete del notebook. Si ya están instalados, vuelve a ejecutarla solo si hace falta.

In [ ]:
%pip install -r requirements.txt

## 3. Clave y configuración del modelo

- **Colab:** crea un secreto llamado `OPENAI_API_KEY` en el panel de secretos y permite el acceso desde este notebook. Si la variable de entorno ya está definida, tiene prioridad.
- **Local:** define `OPENAI_API_KEY` como variable de entorno o configúrala en `.env` en la raíz del repositorio. No escribas la clave en una celda ni la compartas en salidas o commits.

Por defecto este notebook usa OpenAI y `gpt-5.4-nano`. Para NVIDIA configura `XQ_PROVEEDOR=nvidia`, `NVIDIA_API_KEY` y un `XQ_MODELO` disponible; el anterior `meta/llama-3.3-70b-instruct` fue retirado. Si cambias de proveedor, usa su clave y una URL correspondiente. El modelo queda registrado en los archivos. `XQ_TEMPERATURA` y `XQ_URL_BASE` son opcionales; usa únicamente un servidor de confianza, pues recibirá la clave y las preguntas.

In [ ]:
from director.llm import cargar_config
from dotenv import load_dotenv

load_dotenv(raiz / '.env', override=False)
os.environ.setdefault('XQ_PROVEEDOR', 'openai')
config = cargar_config()  # También carga .env local, si existe.
if not hasattr(config, 'proveedor'):
    raise RuntimeError('La copia del repositorio no incluye soporte para XQ_PROVEEDOR. Usa la rama o commit actualizado y reinicia el kernel.')
variable_clave = 'OPENAI_API_KEY' if config.proveedor == 'openai' else 'NVIDIA_API_KEY'
if not os.environ.get(variable_clave, '').strip() and EN_COLAB:
    from google.colab import userdata
    clave = userdata.get(variable_clave)
    if clave and clave.strip():
        os.environ[variable_clave] = clave
    del clave
if not os.environ.get(variable_clave, '').strip():
    raise RuntimeError(f'Falta {variable_clave}: configúrala en el entorno, en .env local o en los secretos de Colab.')

print('Clave configurada; su valor no se muestra.')
print('Proveedor:', config.proveedor, '| Modelo:', config.modelo, '| Temperatura:', config.temperatura)
if config.modelo == 'meta/llama-3.3-70b-instruct':
    print('AVISO: el modelo predeterminado fue retirado; configura XQ_MODELO y vuelve a ejecutar esta celda.')

## 4. Preparar la evaluación

Se usan los casos definidos en `experiments/casos.py` y `calcular_esperado`, que consulta las herramientas directamente sin llamar al modelo. La semilla 42 reproduce las referencias del README; la temperatura no garantiza respuestas idénticas entre ejecuciones. Cada celda guarda los archivos en una carpeta nueva con fecha y hora UTC para no sobrescribir ejecuciones anteriores. En Colab descarga los archivos que quieras conservar antes de terminar la sesión.

In [ ]:
import json
import tempfile
from datetime import datetime, timezone
from experiments.casos import CASOS, ejecutar_casos

SEMILLA = 42
carpeta_resultados = raiz / 'experiments' / 'resultados'

def ejecutar_y_mostrar(caso_id):
    caso = next(c for c in CASOS if c['id'] == caso_id)
    carpeta_resultados.mkdir(parents=True, exist_ok=True)
    prefijo = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_') + caso_id + '_'
    salida = Path(tempfile.mkdtemp(prefix=prefijo, dir=carpeta_resultados))
    print(f'Ejecutando {caso_id}; salida: {salida}', flush=True)
    ruta_json, ruta_md = ejecutar_casos(semilla=SEMILLA, salida=salida, config=config, casos=[caso])
    informe = json.loads(ruta_json.read_text(encoding='utf-8'))
    resultado = informe['casos'][0]
    print('\nPregunta:', resultado['pregunta'])
    print('Herramientas esperadas:', ', '.join(resultado['herramientas_esperadas']))
    print('Resultado esperado:', json.dumps(resultado['resultado_esperado'], ensure_ascii=False, indent=2))
    print('\nRegistro completo de herramientas:')
    if not resultado['llamadas']:
        print('Ninguna herramienta ejecutada.')
    for numero, llamada in enumerate(resultado['llamadas'], 1):
        print(f'{numero}. {llamada["herramienta"]}')
        print('   Argumentos:', json.dumps(llamada['argumentos'], ensure_ascii=False))
        print('   Resultado:', llamada['resultado'])
        print('   Error:', llamada['error'], '| Segundos:', llamada['segundos'])
    print('\nRespuesta final:', resultado['respuesta'])
    print('Llamadas:', resultado['numero_llamadas'], '| Límite alcanzado:', resultado['limite_alcanzado'])
    print('Latencia (s):', resultado['latencia_segundos'])
    print('Evidencia JSON:', ruta_json)
    print('Evidencia Markdown:', ruta_md)
    return resultado

print('Semilla:', SEMILLA, '| Modelo:', config.modelo)

## 5. Ejecutar los cinco casos

Compara en cada celda la respuesta y los argumentos/resultados de las herramientas con la referencia mostrada. La selección de herramientas puede variar entre modelos; comprueba el contenido y las acciones, no solo el nombre de la herramienta. Si la API falla, la celda muestra el error y los archivos de los casos anteriores permanecen guardados.

En una ejecución previa con `gpt-5.4-nano`, prompt 1 y semilla 42, C1, C2 y C5 fueron correctos; C3 sugirió un destino pese a no tener historial y C4 no comparó la distancia antes y después del avance. Consulta la sección 6 del README para el análisis y las trazas. Tus resultados pueden variar.

### C1 — Sala y distancia (habitual)
Con semilla 42 se espera que el tripulante esté en Bodega y que el alienígena esté a 15 pasos.

In [ ]:
c1 = ejecutar_y_mostrar('C1')

### C2 — Visión (habitual)
Con semilla 42 el alienígena no ve al tripulante; la distancia calculada es de 15 casillas.

In [ ]:
c2 = ejecutar_y_mostrar('C2')

### C3 — Predicción de destino (habitual)
Al comenzar la partida, el predictor devuelve Bodega con confianza 0 y sin ticks analizados. La respuesta no debería presentar esa estimación como un destino seguro.

In [ ]:
c3 = ejecutar_y_mostrar('C3')

### C4 — Pista y avance (varias herramientas y acción)
Con semilla 42 se espera una pista a Bodega, 10 ticks de avance y un acercamiento de 15 a 6 pasos; quedan 2 pistas y no hubo captura. Comprueba en la traza que se midió la distancia antes y después y que la respuesta indica si se acercó.

In [ ]:
c4 = ejecutar_y_mostrar('C4')

### C5 — Sala inexistente (límite)
Enfermería no existe en el mapa. Se espera que el agente informe el error y las salas válidas sin inventar una distancia.

In [ ]:
c5 = ejecutar_y_mostrar('C5')

### C6 — Caso adicional: límite de llamadas
Esta celda opcional consulta las distancias a las ocho salas. Con límite de seis herramientas se espera que el agente comunique el límite y no invente distancias faltantes. En una ejecución anterior usó una llamada de estado y cinco de ruta: informó cinco distancias correctas y declaró que faltaban Bodega, Reactor y Hangar; el resultado puede variar.

In [ ]:
c6 = ejecutar_y_mostrar('C6')

## 6. Comparación y conservación de evidencia

Revisa para cada caso la respuesta final, la traza y `resultado_esperado`; completa manualmente `valoracion` y `correcto` en el JSON generado. Esos campos quedan inicialmente vacíos y el Markdown generado también indica que la valoración está pendiente; si se edita uno, el otro no se sincroniza automáticamente. Anota cualquier dato incorrecto, omisión o limitación y una mejora posible. C6 se valora por separado según el comportamiento ante el límite, no como una respuesta completa a las ocho distancias.

Si una respuesta depende de una herramienta incorrecta o una acción no se realizó, documéntalo aunque el texto final parezca acertado. No se asigna automáticamente un valor booleano a respuestas en lenguaje natural.